# Step 9: training

This notebook goes with step 9 of `vak`. It trains a tiny GPT on two short stories, small enough to watch every
number. `uv run vak train` trains the real GPT on 50,000 stories.

Before this: step 8 builds the GPT, untrained. After this: step 10 makes its writing faster with a KV cache.

| Section | Idea | Used in |
|---|---|---|
| 1 | Inputs and targets: the same window, moved one token on | `vak train` |
| 2 | One training step: the loss, its gradient, and AdamW | `vak train` |
| 3 | The loop: the loss falls to what the stories allow | `vak train` |
| 4 | The paired difference: two models on the same stories | `vak train` |

## 1. Inputs and targets

Training needs a right answer at every place. The stories are joined into one long **stream** of ids, with one end
token between two stories. A **window** of ids from the stream is the input, and the same window moved one token
on is the **target**: at each place, the model must predict the next id. Here are two stories, as made-up ids.

In [1]:
import torch
from vak.train import batch, join

stream = join([[0, 5, 6, 0], [0, 7, 8, 9, 0]])  # two stories, each framed by the end token, 0
print("the stream:", stream.tolist())
inputs, targets = batch(stream, size=3, context=3, generator=torch.Generator().manual_seed(1))
for row_in, row_out in zip(inputs.tolist(), targets.tolist()):
    print("input", row_in, "  target", row_out)

the stream: [0, 5, 6, 0, 7, 8, 9, 0]
input [0, 5, 6]   target [5, 6, 0]
input [7, 8, 9]   target [8, 9, 0]
input [7, 8, 9]   target [8, 9, 0]


**What you see**

The two stories share one end token: `[0, 5, 6, 0, 7, 8, 9, 0]`. Each target row is its input row moved one id on:
the input `[0, 5, 6]` has the `target [5, 6, 0]`. So one window of 3 ids asks 3 questions with known answers. A
window can cross from one story into the next, through the end token: the model learns that a new story starts
after it. With a stream this short, two of the three windows happen to land on the same place.

Used in: `uv run vak train` draws 64 windows of 128 ids at each step, from a stream of 12.6 million ids.

## 2. One training step

One step: run the model on the inputs, and score its guesses with the **cross-entropy loss**: minus the log of the
chance it gave each target, averaged. Then `loss.backward()` finds the **gradient**: for every weight, which way
to move it to lower the loss. **AdamW** moves each weight a small step that way. Here is one step on a small GPT,
with the loss before and after, on the same batch.

In [2]:
import math

from torch.nn import functional as F
from medha.model import GPT, Config

torch.manual_seed(0)
model = GPT(Config(vocab_size=10, context=3, width=16, heads=2, layers=1))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
inputs, targets = batch(stream, size=4, context=3, generator=torch.Generator().manual_seed(0))
for step in range(3):
    logits = model(inputs)
    loss = F.cross_entropy(logits.reshape(-1, 10), targets.reshape(-1))
    print(f"step {step}: loss {loss.item():.3f}")
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print("an even guess over 10 ids:", round(math.log(10), 3))

step 0: loss 2.317
step 1: loss 2.106
step 2: loss 1.989
an even guess over 10 ids: 2.303


**What you see**

Before any step, the loss is `2.317`, close to ln 10 = `2.303`: an even guess over 10 ids. Each step moves every
weight a little, and the loss on the same batch falls, to `2.106` and then `1.989`.

Used in: `uv run vak train` runs 5,000 such steps, with the learning rate 0.001.

## 3. The loop

The loop repeats the step on fresh windows. Here a tiny GPT learns two stories that differ in one word: "The cat
sat on the mat." and "The dog sat on the mat.". `train` is the same function the command runs. Then
`story_losses` scores each token of one story: the loss of every next token, as the pair counts score them.

In [3]:
from medha.tokenizer import END, train_bpe
from vak.train import story_losses, train

stories = ["The cat sat on the mat.", "The dog sat on the mat."]
tokenizer = train_bpe(stories * 10, vocab_size=300)
end = tokenizer.token_to_id(END)
framed = []  # each story with the end token before and after it
for text in stories * 100:
    framed.append([end] + tokenizer.encode(text).ids + [end])
stream = join(framed)
story = framed[0]
torch.manual_seed(0)
model = GPT(Config(vocab_size=300, context=16, width=32, heads=2, layers=2))
before = story_losses(model, story)
train(model, stream, steps=300, tokenizer=tokenizer, seed=0)
after = story_losses(model, story)
pieces = ["<|endoftext|>"] + tokenizer.encode(stories[0]).tokens + ["<|endoftext|>"]
for piece, loss_before, loss_after in zip(pieces[1:], before, after):
    print(f"{piece:15} loss before {loss_before:.3f}   after {loss_after:.3f}")

The             loss before 5.745   after 0.057
Ġcat            loss before 5.777   after 0.848
Ġsat            loss before 5.831   after 0.052
Ġon             loss before 5.701   after 0.052
Ġthe            loss before 5.631   after 0.051
Ġmat            loss before 5.665   after 0.050
.               loss before 5.558   after 0.055
<|endoftext|>   loss before 5.863   after 0.055


**What you see**

Before training, every token costs about ln 300 = 5.70, an even guess: `5.745` for "The". After 300 steps, almost
every token costs about 0.05: the model is sure of it. One token stays high, `0.848`: the animal, "Ġcat". The
stories have a cat and a dog equally often, so the best guess there is a coin flip, which costs ln 2 = 0.693.
Training lowers the loss toward what the stories allow, and no further.

Used in: `uv run vak train` runs `train` for 5,000 steps on the real stream, then `story_losses` on each of the
2,000 held-out stories.

## 4. The paired difference

Two models are best compared on the same stories. Some stories are hard for every model, so the two losses rise and
fall together. The **paired difference** takes our loss minus theirs on each story, then bootstraps over stories:
it draws the same stories for both, so how hard a story is cancels out. Here, model A is 0.5 better than model B on
every story, and the stories are of two kinds, easy and hard.

In [4]:
import random
from vak.loss import held_out_loss, paired_difference

draw = random.Random(0)
model_a = []
model_b = []
for number in range(40):
    hard = 3.0 if number % 2 == 0 else 5.0  # half the stories are easy, half are hard
    losses_b = []
    losses_a = []
    for _ in range(100):
        losses_b.append(hard + draw.gauss(0, 0.05))
        losses_a.append(hard - 0.5 + draw.gauss(0, 0.05))
    model_b.append(losses_b)
    model_a.append(losses_a)
a = held_out_loss(model_a, seed=0)
b = held_out_loss(model_b, seed=0)
gap = paired_difference(model_a, model_b, seed=0)
print(f"model A    {a.mean:.3f}, 95% interval {a.low:.3f} to {a.high:.3f}")
print(f"model B    {b.mean:.3f}, 95% interval {b.low:.3f} to {b.high:.3f}")
print(f"A minus B {gap.mean:+.3f}, 95% interval {gap.low:+.3f} to {gap.high:+.3f}")

model A    3.500, 95% interval 3.199 to 3.800
model B    4.001, 95% interval 3.700 to 4.301
A minus B -0.501, 95% interval -0.503 to -0.499


**What you see**

Alone, each interval spans about 0.6, `3.199 to 3.800` and `3.700 to 4.301`, and the two overlap: from them, A
might not look better. The paired difference, `-0.501`, has the interval `-0.503 to -0.499`: A is better on every
story, by the same amount.

Used in: `uv run vak train` reports the GPT minus the pair counts this way, on the 2,000 held-out stories.

## Recap

| Idea | Remember | Used next |
|---|---|---|
| Targets | the input window moved one token on: one question per place | every training step |
| One step | loss, `loss.backward()` for the gradient, then AdamW moves each weight a little | 5,000 steps in `vak train` |
| The loop | the loss falls toward what the data allows: a coin flip stays at ln 2 | the held-out loss of the trained GPT |
| Paired difference | our loss minus theirs, story by story, with one bootstrap over stories | the GPT against the pair counts |